# Casos de Dengue em Porto Alegre — 2025

Análise focada nos casos confirmados de dengue do município de Porto Alegre ao longo de 2025.

**Fonte:** SINAN/OpenDataSUS — arquivos `DENGBR25.csv` e `DENGBR26.csv`  
**Filtro:** `ID_MUNICIP == 431490` e `CLASSI_FIN ∈ [10, 11, 12]`  
**Período de interesse:** semana epidemiológica 1/2025 a 52/2025


In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

PASTA_BASE  = Path('../Bases de dados/bases_oficiais_opendatasus')
PASTA_POA   = Path('../Bases de dados/base_oficial_filtrada_poa')
CODIGO_POA  = 431490
CLASSI_CONF = [10, 11, 12]


## Carregamento dos Dados

Lê os arquivos `DENGBR25.csv` e `DENGBR26.csv` em chunks, filtrando apenas Porto Alegre + classificação confirmada.  
O `DENGBR26.csv` é incluído pois pode conter notificações de 2025 registradas com atraso.


In [ ]:
def filtrar_poa_chunks(csv_path):
    chunks = []
    for chunk in pd.read_csv(csv_path, encoding='latin1', low_memory=False, chunksize=100_000):
        chunk['ID_MUNICIP'] = pd.to_numeric(chunk['ID_MUNICIP'], errors='coerce')
        chunk['CLASSI_FIN'] = pd.to_numeric(chunk['CLASSI_FIN'], errors='coerce')
        filtrado = chunk[
            (chunk['ID_MUNICIP'] == CODIGO_POA) &
            (chunk['CLASSI_FIN'].isin(CLASSI_CONF))
        ]
        if not filtrado.empty:
            chunks.append(filtrado)
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

dfs = []
for arquivo in ['DENGBR25.csv', 'DENGBR26.csv']:
    path = PASTA_BASE / arquivo
    df = filtrar_poa_chunks(path)
    print(f'{arquivo}: {len(df):,} casos de POA encontrados')
    dfs.append(df)

df_raw = pd.concat(dfs, ignore_index=True)

# Converter datas e manter apenas 2025
df_raw['DT_SIN_PRI'] = pd.to_datetime(df_raw['DT_SIN_PRI'], errors='coerce')
df_raw['DT_NOTIFIC']  = pd.to_datetime(df_raw['DT_NOTIFIC'],  errors='coerce')

df_2025 = df_raw[df_raw['DT_SIN_PRI'].dt.year == 2025].copy()

print(f'\nTotal 2025 (por DT_SIN_PRI): {len(df_2025):,} casos')
print(f'Período: {df_2025["DT_SIN_PRI"].min().date()} até {df_2025["DT_SIN_PRI"].max().date()}')


## Visão Geral


In [ ]:
print('=== Classificação Final ===')
print(df_2025['CLASSI_FIN'].map({10: 'Dengue', 11: 'Com sinais de alarme', 12: 'Dengue grave'}).value_counts().to_string())

print('\n=== Critério de Confirmação ===')
print(df_2025['CRITERIO'].map({1: 'Laboratório', 2: 'Clínico-epidemiológico', 3: 'Clínico'}).value_counts().to_string())

print('\n=== Hospitalização ===')
print(df_2025['HOSPITALIZ'].map({1: 'Sim', 2: 'Não', 9: 'Ignorado'}).value_counts().to_string())

print('\n=== Evolução ===')
print(df_2025['EVOLUCAO'].map({1: 'Cura', 2: 'Óbito dengue', 3: 'Óbito outras causas', 4: 'Óbito investigação', 9: 'Ignorado'}).value_counts().to_string())


## Distribuição Mensal


In [ ]:
mensal = (
    df_2025
    .assign(mes=df_2025['DT_SIN_PRI'].dt.month)
    .groupby('mes')
    .size()
    .reindex(range(1, 13), fill_value=0)
    .reset_index(name='casos')
)
mensal['data'] = pd.to_datetime(mensal['mes'].apply(lambda m: f'2025-{m:02d}-01'))

MESES_PT = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(mensal['mes'], mensal['casos'], color='steelblue')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MESES_PT)
ax.set_title('Casos Confirmados de Dengue por Mês — Porto Alegre 2025')
ax.set_xlabel('Mês')
ax.set_ylabel('Número de casos')

# Anotar valor em cada barra
for _, row in mensal.iterrows():
    if row['casos'] > 0:
        ax.text(row['mes'], row['casos'] + 20, f"{int(row['casos']):,}",
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print(mensal[['mes', 'casos']].assign(mes=mensal['mes'].map(dict(enumerate(MESES_PT, 1)))).to_string(index=False))


## Distribuição por Semana Epidemiológica


In [ ]:
# SEM_PRI formato AAAASS → extrair semana
df_2025['sem'] = pd.to_numeric(df_2025['SEM_PRI'], errors='coerce') % 100

semanal = (
    df_2025
    .groupby('sem')
    .size()
    .reindex(range(1, 53), fill_value=0)
    .reset_index(name='casos')
)

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(semanal['sem'], semanal['casos'], color='steelblue', width=0.8)
ax.set_title('Casos Confirmados de Dengue por Semana Epidemiológica — Porto Alegre 2025')
ax.set_xlabel('Semana Epidemiológica')
ax.set_ylabel('Número de casos')
ax.set_xticks(range(1, 53))
plt.tight_layout()
plt.show()
